In [8]:
import pandas as pd
import numpy as np

# 1. Load raw dataset
df = pd.read_csv('ONLINE_FOOD_DELIVERY_ANALYSIS.csv')
print("Initial Dataset Shape:", df.shape)
print("\nMissing Values Before Cleaning:\n", df.isnull().sum()[df.isnull().sum() > 0])


Initial Dataset Shape: (100000, 25)

Missing Values Before Cleaning:
 Customer_Age           50093
Customer_Gender        24856
City                   16726
Area                   16685
Cuisine_Type           16885
Order_Date              1014
Order_Time              1998
Delivery_Time_Min      33359
Distance_km            33470
Order_Value            33327
Discount_Applied       16715
Final_Amount           55697
Payment_Mode           19911
Cancellation_Reason    90969
Delivery_Rating        16523
Peak_Hour              32962
dtype: int64


In [9]:
# 2. Standardizing Categorical Values (Requirement 4)
cat_cols = ['Customer_Gender', 'City', 'Area', 'Cuisine_Type', 'Payment_Mode', 'Order_Status', 'Cancellation_Reason']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()
        df[col] = df[col].replace({'Nan': np.nan, 'None': np.nan, '': np.nan})

# Impute missing categorical values using Mode (Requirement 1)
impute_cat_cols = ['Customer_Gender', 'City', 'Area', 'Cuisine_Type', 'Payment_Mode']
for col in impute_cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [10]:
# 3. Correcting Invalid Values (Requirement 3)
# Cap ratings at 5.0 and floor at 1.0
df['Restaurant_Rating'] = df['Restaurant_Rating'].clip(lower=1.0, upper=5.0)
df['Delivery_Rating'] = df['Delivery_Rating'].clip(lower=1.0, upper=5.0)

# Replace negative Profit_Margin with median of valid (>= 0) profit margins
valid_margin_median = df.loc[df['Profit_Margin'] >= 0, 'Profit_Margin'].median()
df.loc[df['Profit_Margin'] < 0, 'Profit_Margin'] = valid_margin_median

# Ensure Discount_Applied is non-negative
df['Discount_Applied'] = df['Discount_Applied'].clip(lower=0.0)


In [11]:
# 4. Handling Missing Numerical Values using Median (Requirement 1)
num_cols = ['Customer_Age', 'Delivery_Time_Min', 'Distance_km', 'Order_Value', 'Discount_Applied', 'Restaurant_Rating']
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Recalculate Final_Amount safely so discounts never create negative totals
df['Final_Amount'] = (df['Order_Value'] - df['Discount_Applied']).clip(lower=0.0)


In [12]:
# 5. Ensuring Logical Consistency (Requirement 5)
# Delivered orders must have Cancellation_Reason = 'None'
df.loc[df['Order_Status'] == 'Delivered', 'Cancellation_Reason'] = 'None'

# Cancelled orders with missing reason get 'Unspecified'
df.loc[(df['Order_Status'] == 'Cancelled') & (df['Cancellation_Reason'].isnull()), 'Cancellation_Reason'] = 'Unspecified'

# Cancelled orders cannot have a Delivery_Rating
df.loc[df['Order_Status'] == 'Cancelled', 'Delivery_Rating'] = np.nan

# Delivered orders with missing Delivery_Rating get imputed with median delivered rating
delivered_rating_median = df.loc[df['Order_Status'] == 'Delivered', 'Delivery_Rating'].median()
df.loc[(df['Order_Status'] == 'Delivered') & (df['Delivery_Rating'].isnull()), 'Delivery_Rating'] = delivered_rating_median


In [13]:
# 6. Capping Outliers using IQR Method (Requirement 2)
for col in ['Delivery_Time_Min', 'Order_Value']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = max(0, Q1 - 1.5 * IQR)
    upper_bound = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

# Clean date format and remove exact duplicates
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df = df.drop_duplicates()

print("Cleaned Dataset Shape:", df.shape)
print("\nRemaining Nulls (Only Delivery_Rating for Cancelled orders should be null):\n", df.isnull().sum()[df.isnull().sum() > 0])


Cleaned Dataset Shape: (100000, 25)

Remaining Nulls (Only Delivery_Rating for Cancelled orders should be null):
 Order_Date          1014
Order_Time          1998
Delivery_Rating    15036
Peak_Hour          32962
dtype: int64


In [15]:
# Fix remaining operational nulls before saving
df['Order_Date'] = df['Order_Date'].ffill().bfill()
df['Order_Time'] = df['Order_Time'].fillna(df['Order_Time'].mode()[0])
df['Peak_Hour'] = df['Peak_Hour'].fillna(False).astype(bool)

print("Final Null Check (Only Cancelled Delivery_Rating should remain):")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Overwrite 01_cleaned_data.csv with zero unexpected nulls
df.to_csv('01_cleaned_data.csv', index=False)
print("Updated 01_cleaned_data.csv successfully!")


Final Null Check (Only Cancelled Delivery_Rating should remain):
Delivery_Rating    15036
dtype: int64
Updated 01_cleaned_data.csv successfully!
